In [38]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import ast
from collections import defaultdict
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import copy
import importlib
import time
import json
import requests
import unicodedata



import scripts.cleaning as cleaning
import scripts.knn_class as knn_class
import scripts.eval as eval

In [39]:
importlib.reload(knn_class)

<module 'scripts.knn_class' from 'c:\\Projects\\F1-Predictor\\scripts\\knn_class.py'>

In [40]:
filename = "2025_F1_Races - 20263.csv"
df_dict = cleaning.import_and_clean(filename)
# df_dict
df = cleaning.format_testing(df_dict)
# df.to_csv(f"testing2026.csv", index = False)
df

,driver,prev5_avg,prev5_points,points_behind_leader,current_points,percent_of_max,position,race_number
0,A. AntonelliMercedes,0.0,[],7.0,18.0,0.720000,2,1
1,G. RussellMercedes,0.0,[],0.0,25.0,1.000000,1,1
2,L. HamiltonFerrari,0.0,[],13.0,12.0,0.480000,4,1
3,C. LeclercFerrari,0.0,[],10.0,15.0,0.600000,3,1
4,L. NorrisMcLaren,0.0,[],15.0,10.0,0.400000,5,1
...,...,...,...,...,...,...,...,...
193,F. AlonsoAston Martin Racing,0.2,"[0.0, 0.0, 1.0, 0.0, 0.0]",178.0,1.0,0.005587,18,9
194,N. HulkenbergAudi,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",179.0,0.0,0.000000,19,9
195,V. BottasCadillac F1 Team,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",179.0,0.0,0.000000,20,9
196,S. PerezCadillac F1 Team,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",179.0,0.0,0.000000,21,9


In [41]:
df["prev5_points"] = (
    df.groupby("driver", sort=False)["prev5_points"]
      .shift(-1)
)

df["prev5_points"] = df["prev5_points"].apply(
    lambda x: x if isinstance(x, list) else []
)

df["prev5_avg"] = (
    df.groupby("driver", sort=False)["prev5_avg"]
      .shift(-1)
)

# df["prev5_avg"] = df["prev5_avg"].apply(
#     lambda x: x if isinstance(x, list) else []
# )
df

,driver,prev5_avg,prev5_points,points_behind_leader,current_points,percent_of_max,position,race_number
0,A. AntonelliMercedes,18.0,[18.0],7.0,18.0,0.720000,2,1
1,G. RussellMercedes,25.0,[25.0],0.0,25.0,1.000000,1,1
2,L. HamiltonFerrari,12.0,[12.0],13.0,12.0,0.480000,4,1
3,C. LeclercFerrari,15.0,[15.0],10.0,15.0,0.600000,3,1
4,L. NorrisMcLaren,10.0,[10.0],15.0,10.0,0.400000,5,1
...,...,...,...,...,...,...,...,...
193,F. AlonsoAston Martin Racing,NaN,[],178.0,1.0,0.005587,18,9
194,N. HulkenbergAudi,NaN,[],179.0,0.0,0.000000,19,9
195,V. BottasCadillac F1 Team,NaN,[],179.0,0.0,0.000000,20,9
196,S. PerezCadillac F1 Team,NaN,[],179.0,0.0,0.000000,21,9


In [42]:
# df = df[df["race_number"] == 8]
# result = df.set_index("driver").to_dict(orient="index")
# with open("drivers.json", "w") as f:
#     json.dump(result, f, indent=4)

In [43]:
def update_predictions2(completed_race, year = 2026, year_races = 22, sims = 10000):
    knn_sim = knn_class.KNNSeasonSimulator()
    knn_sim.set_neighbors(year)

    # change this to read json and covert to pandas
    # with open('training_data/current_standings.json', 'r', encoding='utf-8') as file:
    #     data = json.load(file)
    # df_eval = pd.DataFrame.from_dict(data, orient='index').reset_index().rename(columns={'index': 'driver'})
    # df_eval['prev5_points'] = df_eval['prev5_points'].apply(ast.literal_eval)
    df_eval = df

    all_sims = []
    num_races = year_races
    # print(df_eval)
    df_eval_small = (df_eval[df_eval.race_number == completed_race]).set_index("driver").to_dict(orient='index')
    res = []
    for j in range(sims):
        if j % 1000 == 0:
            print(j, completed_race)
        df_sim = copy.deepcopy(df_eval_small)
        # print(df_sim)
        # print(completed_race)
        df_sim = knn_sim.simulate_season(df_sim, num_races)

        res.append({
            d: df_sim[d]["current_points"]
            for d in df_sim  
        })

    all_sims.append(res)

    data = defaultdict(lambda: defaultdict(int))
    for i in range(sims):
        d = all_sims[0][i]

        sort = dict(sorted(d.items(), key=lambda item: item[1], reverse=True))
        c = 1
        for k,v in sort.items():
            if c == 1:
                data[k]["1st"] += 100 * 1/sims
            if c <= 3:
                data[k]["Podium"] += 100 * 1/sims
            if c <= 5:
                data[k]["Top 5"] += 100 * 1/sims
            if c <= 10:
                data[k]["Top 10"] += 100 * 1/sims
            
            data[k]["points"] += v/sims
            
            c += 1

    # url = "https://api.jolpi.ca/ergast/f1/2026/driverStandings.json"

    # response = requests.get(url)
    
    # data2 = response.json()["MRData"]["StandingsTable"]
    
    # data["completed race"] = completed_race + 1
    # data["point total"] = total_points(data2["StandingsLists"][0]["DriverStandings"])

    # json_string = json.dumps(data, indent=4)
    # with open("json/predictions.json", "w") as file:
    #     file.write(json_string)
    return data

In [44]:
d = {}
for i in range(5, 9):
    d[str(i)] = update_predictions2(i, year = 2026, year_races = 22, sims = 10000)


0 5
1000 5
2000 5
3000 5
4000 5
5000 5
6000 5
7000 5
8000 5
9000 5
0 6
1000 6
2000 6
3000 6
4000 6
5000 6
6000 6
7000 6
8000 6
9000 6
0 7
1000 7
2000 7
3000 7
4000 7
5000 7
6000 7
7000 7
8000 7
9000 7
0 8
1000 8
2000 8
3000 8
4000 8
5000 8
6000 8
7000 8
8000 8
9000 8


In [45]:
json_string = json.dumps(d, indent=4)
with open("json/all_predictions.json", "w") as file:
    file.write(json_string)
print(json_string)

{
    "5": {
        "A. AntonelliMercedes": {
            "1st": 96.74000000001259,
            "Podium": 99.99000000001425,
            "Top 5": 100.00000000001425,
            "Top 10": 100.00000000001425,
            "points": 460.1664000000008
        },
        "G. RussellMercedes": {
            "Podium": 82.28000000000519,
            "Top 5": 99.18000000001383,
            "Top 10": 100.00000000001425,
            "points": 305.28079999999926,
            "1st": 2.6399999999999877
        },
        "L. HamiltonFerrari": {
            "Podium": 42.83000000000005,
            "Top 5": 93.29000000001082,
            "Top 10": 100.00000000001425,
            "points": 240.54990000000072,
            "1st": 0.16
        },
        "C. LeclercFerrari": {
            "Top 5": 96.02000000001222,
            "Top 10": 100.00000000001425,
            "points": 256.2310000000014,
            "Podium": 55.759999999997476,
            "1st": 0.45000000000000023
        },
        "L. Norr